# 1. Package Imports section

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window

# 2. Dataset Config

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Load_Profiles")

rename_cols = {
    "Timestamp": "timestamp",
    "EislebenDrive": "Eisleben Drive",
    "GrassyPark": "Grassy Park",
    "JanSmuts": "Jan Smuts",
    "LochRoad": "Loch Road",
    "MitchellsPlain": "Mitchells Plain",
    "PelicanPark": "Pelican Park",
    "PiersRoad": "Piers Road",
    "SpineRoad": "Spine Road",
    "SunValley": "Sun Valley",
    "BroadRoad": "Broad Road",
    "CenturyCity": "Century City",
    "City_N": "City North",
    "ConstitutionStreet": "Constitution Street",
    "ElsiesRiver": "Elsies River",
    "GreenPoint": "Green Point",
    "MontagueGardens": "Montague Gardens",
    "MouillePoint": "Mouille Point",
    "RichmondEstate": "Richmond Estate",
    "SeaPoint": "Sea Point",
    "John_Dreyer": "John Dreyer",
    "WilliamGourley": "William Gourley",
    "GordonsBay": "Gordons Bay",
    "SomersetWest": "Somerset West",
    "BellvilleSouth": "Bellville South",
    "Morgen_Gronde": "Morgen Gronde",
    "ParowSouth": "Parow South",
    "SacksCircle": "Sacks Circle",
    "TygerbergHospital": "Tygerberg Hospital",
}

ds_config ={
    "bronze_table": "cpt_utility_catalog.bronze.bronze_substations_raw",
    "silver_table": "cpt_utility_catalog.silver.silver_substations_cleaned",
    "changes":{
        "columns":{
            "columns_to_drop": ["ObjectId"],
            "data_types":{
                "timestamp": "timestamp",
                "substation": "string",
                "load_MVA": "decimal(13, 9)",
            }
        },
        "column_mapping": rename_cols,
        "rename_headers": True,
        "trim": True,
        "cast_data_types": True, 
        "deduped": True,
        "new_totals": True,
        "add_id": True,
        "drop_columns": True,
        "col_cleanse": True,
        "unpivot": True,
        "write_to_table": True,
    }
}

enriched_cols = {
    
}

df_new = spark.read.table(ds_config["bronze_table"])
changes = ds_config["changes"]
col_config = ds_config["changes"]["columns"]

logger.info("Silver layer load profile table configuration loaded")

# 3. Dataset Cleaning

## 3.1 Dropping Columns

In [0]:
if changes["drop_columns"] and isinstance(col_config["columns_to_drop"], list):
    logger.info("Dropping column(s)")
    # dropping columns that won't be needed acccording to config
    df_new = df_new.drop(*col_config["columns_to_drop"])
    logger.info("\t- Column(s) dropped")

## 3.2 Rename Headers

In [0]:
if changes["rename_headers"]:
    logger.info("Renaming headers")

    if isinstance(changes["column_mapping"], dict):

        df_new = df_new.select(
        [F.col(col).alias(changes["column_mapping"].get(col, col)) for col in df_new.columns]
    )

    print(df_new.columns)
    logger.info("\t- Header(s) renamed")

## 3.3 Trim Whitespace

In [0]:
if changes["trim"]:
    logger.info("Trimming whitespace")
    # iterating through all columns and trimming whitespace
    df_new = df_new.select([F.regexp_replace(F.col(col), r"\s+", "").alias(col) for col in df_new.columns])
    logger.info("\t - Whitespace trimmed")

## 3.4 Unpivot Table

In [0]:
if changes["unpivot"]:
    logger.info("Unpivoting table")
        
    exclude_cols = ["timestamp"]

    include_cols = [F.col(col) for col in df_new.columns if col not in exclude_cols]
    # transforming wide table in to a long table
    df_new = df_new.unpivot(
        ids = exclude_cols,
        values = include_cols,
        variableColumnName = "substation",
        valueColumnName = "load_MVA"
    )

    logger.info("\t- Table unpivoted")

## 3.5 Column Cleanse

In [0]:
if changes["col_cleanse"]:
        logger.info("Cleaning column(s)")

        df_new = df_new.withColumn(
            "timestamp", 
            F.regexp_replace(
                F.col("timestamp"),
                r"[+-].*",
                 ""
                )
        )

        # Insert a space between date (yyyy/MM/dd) and time (HH:mm:ss)
        df_new = df_new.withColumn(
            "timestamp",
            F.regexp_replace(
                F.col("timestamp"), 
                r"(\d{4}[/-]\d{2}[/-]\d{2})(\d{2}:\d{2}:\d{2})", 
                r"$1 $2"
            )
        )

        logger.info("\t- Column(s) cleaned")

## 3.6 Cast Data Types

In [0]:
if changes["cast_data_types"]:
    logger.info("Casting Datatype(s)")
    # casting data types according to the config
    df_new = df_new.select([
            F.coalesce(
                F.to_timestamp(F.col(col), "yyyy/MM/dd HH:mm:ss"),
                F.to_timestamp(F.col(col), "dd/MM/yyyy HH:mm:ss"),
            ).alias("timestamp") if col == "timestamp"
            else F.col(col).cast(col_config["data_types"][col]).alias(col) 
            for col in df_new.columns
           
     ])
    logger.info("\t- Datatype(s) casted")

## 3.7 Adding Pk

In [0]:
if changes["add_id"]:
    logger.info("Adding ID(s)")
    # creating primary key column with md5 hash of main columns 
    df_new = df_new.withColumn(
        "id",
        F.md5(
            F.concat_ws(
                "|",
                F.col("timestamp"),
                F.col("substation"),
                F.col("load_MVA"),
            )
        ),
    )
    logger.info("\t- ID(s) added")

## 3.8 Dropping Duplicates

In [0]:
if changes["deduped"]:    
    logger.info("Dropping Duplicate(s)")

    # Check all columns except 'date' for nulls
    excluded_cols = ["timestamp", "id","substations"]
    cols_to_check = [c for c in df_new.columns if c not in exclude_cols]

    # Count how many NULLs exist in each row across all columns
    null_count_expr = sum(
        [F.when(F.col(c).isNull(), 1).otherwise(0) for c in cols_to_check]
    )

    df_with_nulls = df_new.withColumn("null_count", null_count_expr)

    # Create a window grouped by date and dam_name, ordered by null_count ASCENDING least nulls wins.
    # (Lowest null count = rank 1)
    window_spec = Window.partitionBy("timestamp", "substation").orderBy(F.col("null_count").asc())

    # Filter to keep only the best row per date and clean up temporary columns
    df_deduped = (
        df_with_nulls.withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num", "null_count")
    )

    df_new = df_deduped
    logger.info("\t - Duplicate(s) Dropped")

# 4. Writing To Silver Layer

In [0]:
if changes["write_to_table"]:    
    logger.info("Writing load profiles to Silver Table")
    target_table_name = ds_config["silver_table"]

    # 1. Register your cleaned DataFrame as a temporary view for SQL execution
    df_new.createOrReplaceTempView("temp_source_data")

    if spark.catalog.tableExists(target_table_name):
        # 2. Run the native Databricks SQL Merge command
        spark.sql(f"""
            MERGE INTO {target_table_name} AS target
            USING temp_source_data AS source
            ON target.id = source.id
            WHEN MATCHED THEN 
                UPDATE SET *
            WHEN NOT MATCHED THEN 
                INSERT *
        """)
        logger.info("\t - Delta table successfully upserted.")
    else:
        # First-time run: Create the table
        (
            df_new.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table_name)
        )
        logger.info("\t - Target table didnot exist. Created new Delta table.")
    logger.info("\t - Load profiles written to Silver Table")